In [54]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [55]:
import pandas as pd
import numpy as np

# Load embeddings
embeddings = np.load("data/embedded/news_embeddings_final.npy")
news_ids = np.load("data/embedded/news_ids.npy", allow_pickle=True)

# Load behavior data
behaviors = pd.read_csv("data/cleaned/behaviors_clean.csv")

print("Embeddings shape:", embeddings.shape)
print("Number of news IDs:", len(news_ids))
print("Behaviors shape:", behaviors.shape)

behaviors.head()

Embeddings shape: (51282, 768)
Number of news IDs: 51282
Behaviors shape: (5723002, 5)


,user_id,history,news_id,label,category
0,U13740,"['N55189', 'N42782', 'N34694', 'N45794', 'N184...",N55689,1,sports
1,U13740,"['N55189', 'N42782', 'N34694', 'N45794', 'N184...",N35729,0,news
2,U91836,"['N31739', 'N6072', 'N63045', 'N23979', 'N3565...",N20678,0,sports
3,U91836,"['N31739', 'N6072', 'N63045', 'N23979', 'N3565...",N39317,0,news
4,U91836,"['N31739', 'N6072', 'N63045', 'N23979', 'N3565...",N58114,0,autos


In [56]:
news_vector_map = {
    news_id: embeddings[i]
    for i, news_id in enumerate(news_ids)
}

embedding_dim = embeddings.shape[1]

print("Embedding dimension:", embedding_dim)

Embedding dimension: 768


In [57]:
sample_id = news_ids[0]
print(sample_id)
print(news_vector_map[sample_id][:5])

N55528
[-0.01886897  0.06061954  0.03854043  0.00348515  0.04707707]


In [58]:
grouped = behaviors.groupby(["user_id", "history"])

print("Total groups:", len(grouped))

Total groups: 49108


In [59]:
def get_state(history):
    try:
        history = eval(history) if isinstance(history, str) else history
    except:
        return np.zeros(embedding_dim)

    vectors = [news_vector_map[n] for n in history if n in news_vector_map]

    if len(vectors) == 0:
        return np.zeros(embedding_dim)

    return np.mean(vectors, axis=0)

In [60]:
state = get_state(history)
print(state.shape)

(768,)


In [61]:
def extract_candidates(group, news_vector_map):
    news_ids = group["news_id"].values
    labels = group["label"].values

    candidates = []
    valid_labels = []

    for nid, lbl in zip(news_ids, labels):
        if nid in news_vector_map:
            candidates.append(news_vector_map[nid])
            valid_labels.append(lbl)

    if len(candidates) == 0:
        return None, None

    return candidates, np.array(valid_labels)

In [62]:
candidates, labels = extract_candidates(group, news_vector_map)
print(len(candidates), labels)

21 [0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0]


In [63]:
class ContextualBandit:
    def __init__(self, dim, epsilon=0.1):
        self.epsilon = epsilon
        self.theta = np.zeros(dim)

    def predict(self, state, action):
        return np.dot(self.theta, state * action)

    def select_action(self, state, candidates):
        if np.random.rand() < self.epsilon:
            return np.random.randint(len(candidates))

        scores = [self.predict(state, a) for a in candidates]
        return np.argmax(scores)

    def update(self, state, action, reward, lr=0.01):
        self.theta += lr * reward * (state * action)
    
    def get_scores(self, state, candidates):
        return np.array([self.predict(state, a) for a in candidates])

In [64]:
bandit = ContextualBandit(dim=embedding_dim)

total_reward = 0
total_steps = 0

for (user_id, history), group in tqdm(grouped):

    state = get_state(history)

    candidates, labels = extract_candidates(group, news_vector_map)

    if candidates is None:
        continue

    action_idx = bandit.select_action(state, candidates)
    reward = labels[action_idx]

    bandit.update(state, candidates[action_idx], reward)

    total_reward += reward
    total_steps += 1

100%|██████████| 49108/49108 [00:14<00:00, 3349.69it/s]


In [65]:
ctr = total_reward / total_steps
print("CTR:", ctr)

CTR: 0.12849230267980777


In [66]:
from sklearn.metrics import roc_auc_score

def compute_auc(labels, scores):
    if len(set(labels)) < 2:
        return None  # AUC undefined
    return roc_auc_score(labels, scores)

In [67]:
def compute_mrr(labels, scores):
    order = np.argsort(scores)[::-1]
    sorted_labels = labels[order]

    for i, rel in enumerate(sorted_labels):
        if rel == 1:
            return 1 / (i + 1)

    return 0

In [68]:
def compute_ndcg_at_k(labels, scores, k):
    order = np.argsort(scores)[::-1]
    sorted_labels = labels[order][:k]

    # DCG
    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(sorted_labels))

    # IDCG
    ideal_labels = sorted(labels, reverse=True)[:k]
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_labels))

    return dcg / idcg if idcg > 0 else 0

In [69]:
bandit = ContextualBandit(dim=embedding_dim)

total_reward = 0
total_steps = 0

auc_list = []
mrr_list = []
ndcg5_list = []
ndcg10_list = []
ctr_list = []

for (user_id, history), group in tqdm(grouped):

    # ---------------------------
    # State
    # ---------------------------
    state = get_state(history)

    # ---------------------------
    # Candidates
    # ---------------------------
    candidates, labels = extract_candidates(group, news_vector_map)

    if candidates is None:
        continue

    # ---------------------------
    # Scores (for ranking)
    # ---------------------------
    scores = bandit.get_scores(state, candidates)

    # ---------------------------
    # Metrics (BEFORE update)
    # ---------------------------
    auc = compute_auc(labels, scores)
    if auc is not None:
        auc_list.append(auc)

    mrr_list.append(compute_mrr(labels, scores))
    ndcg5_list.append(compute_ndcg_at_k(labels, scores, 5))
    ndcg10_list.append(compute_ndcg_at_k(labels, scores, 10))

    # ---------------------------
    # Bandit decision
    # ---------------------------
    action_idx = bandit.select_action(state, candidates)
    reward = labels[action_idx]

    # ---------------------------
    # Update
    # ---------------------------
    bandit.update(state, candidates[action_idx], reward)

    # ---------------------------
    # CTR tracking
    # ---------------------------
    total_reward += reward
    total_steps += 1
    ctr_list.append(total_reward / total_steps)

100%|██████████| 49108/49108 [00:40<00:00, 1216.60it/s]


In [70]:
print("Final CTR:", ctr_list[-1])
print("Mean AUC:", np.mean(auc_list))
print("Mean MRR:", np.mean(mrr_list))
print("Mean nDCG@5:", np.mean(ndcg5_list))
print("Mean nDCG@10:", np.mean(ndcg10_list))

Final CTR: 0.1324631424615134
Mean AUC: 0.5562185131752394
Mean MRR: 0.27663710884161735
Mean nDCG@5: 0.19123487932253644
Mean nDCG@10: 0.23174852837678422


### UCB

UCB = θᵀx + α * √(xᵀA⁻¹x) <br>
Term	<t>Meaning <br>
θᵀx	predicted <t>reward <br>
√(xᵀA⁻¹x)	<t>uncertainty <br>
α	exploration <t>strength<br>

In [72]:
class LinUCB:
    def __init__(self, dim, alpha=1.0):
        self.dim = dim
        self.alpha = alpha
        
        self.A = np.identity(dim)
        self.b = np.zeros(dim)

    def get_theta(self):
        A_inv = np.linalg.inv(self.A)
        return A_inv @ self.b

    def select_action(self, state, candidates):
        A_inv = np.linalg.inv(self.A)
        theta = A_inv @ self.b

        scores = []

        for action in candidates:
            x = state * action  # interaction feature

            # predicted reward
            mean = np.dot(theta, x)

            # uncertainty
            uncertainty = self.alpha * np.sqrt(x @ A_inv @ x)

            scores.append(mean + uncertainty)

        return np.argmax(scores)

    def get_scores(self, state, candidates):
        A_inv = np.linalg.inv(self.A)
        theta = A_inv @ self.b

        scores = []

        for action in candidates:
            x = state * action
            mean = np.dot(theta, x)
            scores.append(mean)

        return np.array(scores)

    def update(self, state, action, reward):
        x = state * action

        self.A += np.outer(x, x)
        self.b += reward * x

In [76]:
bandit = LinUCB(dim=embedding_dim, alpha=1.0)

In [77]:
auc_list = []
mrr_list = []
ndcg5_list = []
ndcg10_list = []
ctr_list = []

total_reward = 0
total_steps = 0

for (user_id, history), group in tqdm(grouped):

    state = get_state(history)

    candidates, labels = extract_candidates(group, news_vector_map)

    if candidates is None:
        continue

    # ---------------------------
    # Scores for evaluation
    # ---------------------------
    scores = bandit.get_scores(state, candidates)

    auc = compute_auc(labels, scores)
    if auc is not None:
        auc_list.append(auc)

    mrr_list.append(compute_mrr(labels, scores))
    ndcg5_list.append(compute_ndcg_at_k(labels, scores, 5))
    ndcg10_list.append(compute_ndcg_at_k(labels, scores, 10))

    # ---------------------------
    # UCB action selection
    # ---------------------------
    action_idx = bandit.select_action(state, candidates)
    reward = labels[action_idx]

    # ---------------------------
    # Update
    # ---------------------------
    bandit.update(state, candidates[action_idx], reward)

    # ---------------------------
    # CTR tracking
    # ---------------------------
    total_reward += reward
    total_steps += 1
    ctr_list.append(total_reward / total_steps)

100%|██████████| 49108/49108 [59:57<00:00, 13.65it/s]    


In [78]:
print("Final CTR:", ctr_list[-1])
print("Mean AUC:", np.mean(auc_list))
print("Mean MRR:", np.mean(mrr_list))
print("Mean nDCG@5:", np.mean(ndcg5_list))
print("Mean nDCG@10:", np.mean(ndcg10_list))

Final CTR: 0.18304553229616355
Mean AUC: 0.5758808230394664
Mean MRR: 0.3255398702884324
Mean nDCG@5: 0.22253764927473813
Mean nDCG@10: 0.2621685086797749


optimization

In [79]:
def get_state(history):
    try:
        history = eval(history) if isinstance(history, str) else history
    except:
        return np.zeros(embedding_dim)

    vectors = []
    weights = []

    for i, n in enumerate(history[::-1]):  # reverse → recent first
        if n in news_vector_map:
            vectors.append(news_vector_map[n])
            weights.append(1 / (i + 1))  # decay weight

    if len(vectors) == 0:
        return np.zeros(embedding_dim)

    vectors = np.array(vectors)
    weights = np.array(weights)

    weighted_avg = np.average(vectors, axis=0, weights=weights)
    return weighted_avg

In [82]:
import numpy as np

class LinTS:
    def __init__(self, dim, v=1.0):
        self.dim = dim
        self.v = v

        self.A = np.identity(dim)
        self.A_inv = np.identity(dim)
        self.b = np.zeros(dim)

    def get_theta(self):
        return self.A_inv @ self.b

    def sample_theta(self):
        mean = self.get_theta()

        # FAST approximation (important for speed)
        theta_sample = mean + self.v * np.random.randn(self.dim)

        return theta_sample

    def select_action(self, state, candidates):
        theta_sample = self.sample_theta()

        scores = [
            np.dot(theta_sample, state * action)
            for action in candidates
        ]

        return np.argmax(scores)

    def get_scores(self, state, candidates):
        theta = self.get_theta()

        return np.array([
            np.dot(theta, state * action)
            for action in candidates
        ])

    def update(self, state, action, reward):
        x = state * action

        self.A += np.outer(x, x)

        # Sherman-Morrison update (fast inverse)
        A_inv_x = self.A_inv @ x
        self.A_inv -= np.outer(A_inv_x, A_inv_x) / (1 + x @ A_inv_x)

        self.b += reward * x

In [83]:
bandit = LinTS(dim=embedding_dim, v=1.0)

In [84]:
auc_list = []
mrr_list = []
ndcg5_list = []
ndcg10_list = []
ctr_list = []

total_reward = 0
total_steps = 0

In [85]:

for (user_id, history), group in tqdm(grouped):

    # ---------------------------
    # State
    # ---------------------------
    state = get_state(history)

    # ---------------------------
    # Candidates
    # ---------------------------
    candidates, labels = extract_candidates(group, news_vector_map)

    if candidates is None:
        continue

    # ---------------------------
    # Scores (for evaluation)
    # ---------------------------
    scores = bandit.get_scores(state, candidates)

    # ---------------------------
    # Metrics
    # ---------------------------
    auc = compute_auc(labels, scores)
    if auc is not None:
        auc_list.append(auc)

    mrr_list.append(compute_mrr(labels, scores))
    ndcg5_list.append(compute_ndcg_at_k(labels, scores, 5))
    ndcg10_list.append(compute_ndcg_at_k(labels, scores, 10))

    # ---------------------------
    # Thompson Sampling action
    # ---------------------------
    action_idx = bandit.select_action(state, candidates)
    reward = labels[action_idx]

    # ---------------------------
    # Update
    # ---------------------------
    bandit.update(state, candidates[action_idx], reward)

    # ---------------------------
    # CTR tracking
    # ---------------------------
    total_reward += reward
    total_steps += 1
    ctr_list.append(total_reward / total_steps)

100%|██████████| 49108/49108 [04:35<00:00, 178.15it/s]


In [86]:
print("Final CTR:", ctr_list[-1])
print("Mean AUC:", np.mean(auc_list))
print("Mean MRR:", np.mean(mrr_list))
print("Mean nDCG@5:", np.mean(ndcg5_list))
print("Mean nDCG@10:", np.mean(ndcg10_list))

Final CTR: 0.13790013847031032
Mean AUC: 0.5839972028617103
Mean MRR: 0.3279219368516432
Mean nDCG@5: 0.22606741569340014
Mean nDCG@10: 0.2656516110044402


# OPtimization

In [87]:
epsilons = [0.01, 0.05, 0.1, 0.2]

In [88]:
alphas = [0.1, 0.5, 1.0, 2.0]

In [90]:
vs = [0.1, 0.5, 1.0, 2.0]

In [91]:
def run_bandit(bandit, grouped, max_steps=None):

    auc_list, mrr_list, ndcg5_list, ndcg10_list = [], [], [], []
    total_reward, total_steps = 0, 0

    for i, ((user_id, history), group) in enumerate(grouped):

        if max_steps and i >= max_steps:
            break

        state = get_state(history)
        candidates, labels = extract_candidates(group, news_vector_map)

        if candidates is None:
            continue

        scores = bandit.get_scores(state, candidates)

        auc = compute_auc(labels, scores)
        if auc is not None:
            auc_list.append(auc)

        mrr_list.append(compute_mrr(labels, scores))
        ndcg5_list.append(compute_ndcg_at_k(labels, scores, 5))
        ndcg10_list.append(compute_ndcg_at_k(labels, scores, 10))

        action_idx = bandit.select_action(state, candidates)
        reward = labels[action_idx]

        bandit.update(state, candidates[action_idx], reward)

        total_reward += reward
        total_steps += 1

    return {
        "CTR": total_reward / total_steps,
        "AUC": np.mean(auc_list),
        "MRR": np.mean(mrr_list),
        "nDCG@5": np.mean(ndcg5_list),
        "nDCG@10": np.mean(ndcg10_list)
    }

In [92]:
results_eps = []

for eps in epsilons:
    bandit = ContextualBandit(dim=embedding_dim, epsilon=eps)
    res = run_bandit(bandit, grouped, max_steps=5000)
    
    res["epsilon"] = eps
    results_eps.append(res)

pd.DataFrame(results_eps)

,CTR,AUC,MRR,nDCG@5,nDCG@10,epsilon
0,0.1410,0.558811,0.283409,0.191474,0.231955,0.01
1,0.1366,0.558851,0.283643,0.191188,0.231644,0.05
2,0.1380,0.559041,0.284122,0.191400,0.232612,0.10
3,0.1332,0.558923,0.281449,0.189674,0.231404,0.20


In [93]:
results_ts = []

for v in vs:
    bandit = LinTS(dim=embedding_dim, v=v)
    res = run_bandit(bandit, grouped, max_steps=5000)
    
    res["v"] = v
    results_ts.append(res)

pd.DataFrame(results_ts)

,CTR,AUC,MRR,nDCG@5,nDCG@10,v
0,0.1310,0.564680,0.298550,0.200765,0.241598,0.1
1,0.1070,0.564612,0.293320,0.198003,0.238703,0.5
2,0.0912,0.562271,0.289453,0.194271,0.234760,1.0
3,0.0892,0.564228,0.289383,0.195008,0.236649,2.0


In [94]:
class LinUCB:
    def __init__(self, dim, alpha=1.0):
        self.dim = dim
        self.alpha = alpha
        
        self.A = np.identity(dim)
        self.A_inv = np.identity(dim)
        self.b = np.zeros(dim)

    def get_theta(self):
        return self.A_inv @ self.b

    def select_action(self, state, candidates):
        theta = self.get_theta()

        X = np.array([state * a for a in candidates])  # (n, dim)

        mean = X @ theta
        temp = X @ self.A_inv
        uncertainty = np.sqrt(np.sum(temp * X, axis=1))

        ucb_scores = mean + self.alpha * uncertainty

        return np.argmax(ucb_scores)

    def get_scores(self, state, candidates):
        theta = self.get_theta()
        X = np.array([state * a for a in candidates])
        return X @ theta

    def update(self, state, action, reward):
        x = state * action

        self.A += np.outer(x, x)

        A_inv_x = self.A_inv @ x
        self.A_inv -= np.outer(A_inv_x, A_inv_x) / (1 + x @ A_inv_x)

        self.b += reward * x

In [95]:
def get_state(history):
    try:
        history = eval(history) if isinstance(history, str) else history
    except:
        return np.zeros(embedding_dim)

    vectors = []
    weights = []

    for i, n in enumerate(history[::-1]):
        if n in news_vector_map:
            vectors.append(news_vector_map[n])
            weights.append(1 / (i + 1))

    if len(vectors) == 0:
        return np.zeros(embedding_dim)

    return np.average(np.array(vectors), axis=0, weights=np.array(weights))

In [96]:
grouped = behaviors.groupby(["user_id", "history"])

grouped_list = list(grouped)  # convert once

state_cache = {}

for (user_id, history), _ in grouped_list:
    state_cache[(user_id, history)] = get_state(history)

In [97]:
def run_linucb(alpha, grouped_data, max_steps=5000, max_candidates=20):

    bandit = LinUCB(dim=embedding_dim, alpha=alpha)

    auc_list, mrr_list, ndcg5_list, ndcg10_list = [], [], [], []
    total_reward, total_steps = 0, 0

    for i, ((user_id, history), group) in enumerate(tqdm(grouped_data)):

        if i >= max_steps:
            break

        state = state_cache[(user_id, history)]

        candidates, labels = extract_candidates(group, news_vector_map)

        if candidates is None:
            continue

        # limit candidates (speed boost)
        if len(candidates) > max_candidates:
            idx = np.random.choice(len(candidates), max_candidates, replace=False)
            candidates = [candidates[i] for i in idx]
            labels = labels[idx]

        scores = bandit.get_scores(state, candidates)

        auc = compute_auc(labels, scores)
        if auc is not None:
            auc_list.append(auc)

        mrr_list.append(compute_mrr(labels, scores))
        ndcg5_list.append(compute_ndcg_at_k(labels, scores, 5))
        ndcg10_list.append(compute_ndcg_at_k(labels, scores, 10))

        action_idx = bandit.select_action(state, candidates)
        reward = labels[action_idx]

        bandit.update(state, candidates[action_idx], reward)

        total_reward += reward
        total_steps += 1

    return {
        "alpha": alpha,
        "CTR": total_reward / total_steps,
        "AUC": np.mean(auc_list),
        "MRR": np.mean(mrr_list),
        "nDCG@5": np.mean(ndcg5_list),
        "nDCG@10": np.mean(ndcg10_list)
    }

In [98]:
alphas = [0.1, 0.5, 1.0, 2.0]

results = []

for alpha in alphas:
    print(f"\nRunning LinUCB with alpha = {alpha}")
    res = run_linucb(alpha, grouped_list)
    results.append(res)

results_df = pd.DataFrame(results)
print(results_df)


Running LinUCB with alpha = 0.1


 10%|█         | 5000/49108 [00:20<03:01, 243.56it/s]



Running LinUCB with alpha = 0.5


 10%|█         | 5000/49108 [00:20<02:57, 249.16it/s]



Running LinUCB with alpha = 1.0


 10%|█         | 5000/49108 [00:21<03:07, 235.54it/s]



Running LinUCB with alpha = 2.0


 10%|█         | 5000/49108 [00:21<03:08, 233.61it/s]

   alpha     CTR       AUC       MRR    nDCG@5   nDCG@10
0    0.1  0.1268  0.558322  0.245637  0.215702  0.275579
1    0.5  0.1222  0.564133  0.240274  0.212944  0.273643
2    1.0  0.1252  0.559825  0.243663  0.212767  0.273206
3    2.0  0.1248  0.558433  0.241002  0.212876  0.272454
